In [1]:
import pandas as pd
from google.colab import files

uploaded = files.upload()

filename = next(iter(uploaded))
df = pd.read_csv(filename)

print("Dataset loaded successfully!")
print("Number of titles:", len(df))
print(df.columns.tolist())

Saving archive.zip to archive.zip
Dataset loaded successfully!
Number of titles: 8807
['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in', 'description']


In [2]:
# Check that the required columns exist
print("Required columns available:",
      "title" in df.columns and "description" in df.columns)

# Keep titles that have both a title and description
data = df.dropna(
    subset=["title", "description"]
).copy()

# Clean up extra spaces
data["title"] = data["title"].astype(str).str.strip()
data["description"] = (
    data["description"].astype(str).str.strip()
)

# Remove empty titles or descriptions
data = data[
    (data["title"] != "") &
    (data["description"] != "")
].reset_index(drop=True)

print("Original titles:", len(df))
print("Titles ready for AI:", len(data))

display(data[["title", "description"]].head())

Required columns available: True
Original titles: 8807
Titles ready for AI: 8807


,title,description
0,Dick Johnson Is Dead,"As her father nears the end of his life, filmm..."
1,Blood & Water,"After crossing paths at a party, a Cape Town t..."
2,Ganglands,To protect his family from a powerful drug lor...
3,Jailbirds New Orleans,"Feuds, flirtations and toilet talk go down amo..."
4,Kota Factory,In a city of coaching centers known to train I...


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Convert Netflix descriptions into numerical text features
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 2)
)

tfidf_matrix = vectorizer.fit_transform(
    data["description"]
)

print("StreamGap AI text engine is ready!")
print("Titles analysed:", tfidf_matrix.shape[0])
print("Text features created:", tfidf_matrix.shape[1])

StreamGap AI text engine is ready!
Titles analysed: 8807
Text features created: 5000


In [5]:
def recommend_similar(title, number=5):

    # Find the selected title
    matches = data.index[
        data["title"].str.strip().str.casefold()
        == title.strip().casefold()
    ].tolist()

    if not matches:
        print("Title not found. Check the spelling.")
        return

    index = matches[0]

    # Compare its description with every other title
    scores = cosine_similarity(
        tfidf_matrix[index],
        tfidf_matrix
    ).flatten()

    # Exclude the selected title itself
    scores[index] = -1

    # Get the most similar titles
    top_indices = scores.argsort()[::-1]
    top_indices = [
        i for i in top_indices if scores[i] >= 0
    ][:number]

    results = data.iloc[top_indices].copy()

    # Show the similarity score as a percentage
    results["text_similarity_pct"] = (
        scores[top_indices] * 100
    ).round(2)

    columns = [
        "title",
        "type",
        "listed_in",
        "description",
        "text_similarity_pct"
    ]

    columns = [
        col for col in columns
        if col in results.columns
    ]

    return results[columns].reset_index(drop=True)


print("StreamGap recommendation function is ready!")

StreamGap recommendation function is ready!


In [6]:
recommend_similar("Stranger Things")


,title,type,listed_in,description,text_similarity_pct
0,Rowdy Rathore,Movie,"Action & Adventure, Comedies, International Mo...",A con man uncovers a deadly secret and must sa...,41.14
1,The Autopsy of Jane Doe,Movie,"Horror Movies, Independent Movies, Thrillers",A father-son team of small-town coroners perfo...,29.90
2,Dr. Seuss' The Grinch,Movie,"Children & Family Movies, Comedies",A grump with a mean streak plots to bring Chri...,23.80
3,Big Stone Gap,Movie,"Comedies, Romantic Movies",A single middle-aged woman who has lived her w...,22.42
4,Broadchurch,TV Show,"British TV Shows, Crime TV Shows, Internationa...",After a young boy is murdered in the small sea...,22.21


In [7]:
# Combine descriptions, genres, titles, and content type

genre_text = (
    data["listed_in"].fillna("").astype(str)
    if "listed_in" in data.columns
    else pd.Series("", index=data.index)
)

type_text = (
    data["type"].fillna("").astype(str)
    if "type" in data.columns
    else pd.Series("", index=data.index)
)

data["combined_text"] = (
    data["title"].fillna("").astype(str) + " " +
    genre_text + " " +
    genre_text + " " +
    type_text + " " +
    data["description"].fillna("").astype(str)
)

# Rebuild the text similarity model
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    ngram_range=(1, 2)
)

tfidf_matrix = vectorizer.fit_transform(
    data["combined_text"]
)

print("Improved StreamGap model is ready!")
print("Titles analysed:", tfidf_matrix.shape[0])

Improved StreamGap model is ready!
Titles analysed: 8807


In [8]:
recommend_similar("Stranger Things")


,title,type,listed_in,description,text_similarity_pct
0,Nightflyers,TV Show,"TV Horror, TV Mysteries, TV Sci-Fi & Fantasy","With humankind's future at stake, a group of s...",70.26
1,Helix,TV Show,"TV Horror, TV Mysteries, TV Sci-Fi & Fantasy",While investigating a possible outbreak at an ...,67.63
2,Manifest,TV Show,"TV Dramas, TV Mysteries, TV Sci-Fi & Fantasy",When a plane mysteriously lands years after ta...,60.53
3,Chilling Adventures of Sabrina,TV Show,"TV Horror, TV Mysteries, TV Sci-Fi & Fantasy","Magic and mischief collide as half-human, half...",58.86
4,The 4400,TV Show,"TV Dramas, TV Mysteries, TV Sci-Fi & Fantasy",4400 people who vanished over the course of fi...,57.63


In [9]:
!pip -q install gradio

import gradio as gr
import pandas as pd

def streamgap_demo(title):
    title = str(title or "").strip()

    if not title:
        return pd.DataFrame(), "Please enter a Netflix title."

    result = recommend_similar(title, number=5)

    if result is None or result.empty:
        return pd.DataFrame(), (
            "Title not found. Check the spelling "
            "or choose a title from your dataset."
        )

    message = (
        f"Found {len(result)} similar titles. "
        "Scores measure text similarity, not viewing preference."
    )

    return result, message


app = gr.Interface(
    fn=streamgap_demo,
    inputs=gr.Textbox(
        label="Enter a Netflix title",
        placeholder="Example: Stranger Things",
        value="Stranger Things"
    ),
    outputs=[
        gr.Dataframe(label="Recommended titles"),
        gr.Textbox(label="Result")
    ],
    title="StreamGap AI",
    description=(
        "Explore similar Netflix titles using text similarity "
        "across content descriptions and metadata."
    ),
    examples=[
        ["Stranger Things"],
        ["The Crown"],
        ["Money Heist"]
    ]
)

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7936472927f22484c3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
